# Actividad: pronóstico de la temperatura con redes recurrentes (PyTorch)

Ya sabemos pronosticar una serie de tiempo con redes recurrentes. Ahora vamos a aplicar lo aprendido a un problema más rico: **pronosticar la temperatura que hará dentro de 24 horas** a partir de las mediciones de una estación meteorológica durante los últimos 3 días.

Usamos el conjunto de datos de clima de **Jena** (Alemania), registrado por el Instituto Max Planck de Biogeoquímica entre 2009 y 2016: una medición cada 10 minutos de 14 variables (temperatura, presión, humedad, viento, …). A diferencia de los notebooks de la Experiencia, aquí cada paso de tiempo no trae **un** número sino **varios**: es una serie de tiempo **multivariada**.

El notebook ya incluye la carga de los datos, la preparación de las ventanas, un pronóstico de referencia y la clase para entrenar. **Nuestra labor es definir la red recurrente** con la solución que planteamos en la Reflexión (busquen las celdas marcadas con `TODO`), entrenarla y analizar sus resultados.

## Etapa 1: Se carga la base de datos.

In [ ]:
import copy
import urllib.request
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
import matplotlib.pyplot as plt
from tqdm import tqdm

torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Dispositivo:", device)

In [ ]:
data_dir = Path("data")
csv_path = data_dir / "jena_climate_2009_2016.csv"
if not csv_path.exists():
    data_dir.mkdir(exist_ok=True)
    zip_path, _ = urllib.request.urlretrieve(
        "https://storage.googleapis.com/tensorflow/tf-keras-datasets/jena_climate_2009_2016.csv.zip",
        data_dir / "jena_climate_2009_2016.csv.zip")
    with zipfile.ZipFile(zip_path) as f:
        f.extractall(data_dir)

df = pd.read_csv(csv_path)
df["Date Time"] = pd.to_datetime(df["Date Time"], format="%d.%m.%Y %H:%M:%S")
df = df.set_index("Date Time").sort_index()
print(df.shape)
df.head()

Hay una medición cada 10 minutos. Para pronosticar a 24 horas no necesitamos tanto detalle, así que nos quedamos con **una medición por hora** (una de cada 6 filas). Además, el archivo tiene algunos errores de registro en la velocidad del viento (valores de −9999); los reemplazamos por 0.

In [ ]:
df = df.iloc[5::6]   # una fila por hora
for col in ["wv (m/s)", "max. wv (m/s)"]:
    df.loc[df[col] < 0, col] = 0.0
print(df.shape)
df.describe().T[["mean", "std", "min", "max"]]

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 6))
df["T (degC)"].plot(ax=axes[0], title="Temperatura, 2009–2016 (un valor por hora)", grid=True)
df["T (degC)"]["2015-07-01":"2015-07-14"].plot(ax=axes[1], marker=".", title="Dos semanas de julio de 2015", grid=True)
for ax in axes: ax.set_ylabel("°C"); ax.set_xlabel("")
plt.tight_layout()
plt.show()

La temperatura tiene **dos estacionalidades**: una anual (verano e invierno) y una diaria (el día es más caliente que la noche). Por eso las ventanas de 3 días contienen tres ciclos diarios completos.

## Etapa 2: Preparación de los datos

**División en el tiempo.** Como en la Experiencia, dividimos sin mezclar el orden: el primer 70 % de las horas para entrenar, el siguiente 15 % para validar y el último 15 % para probar.

**Normalización.** Las variables tienen escalas muy distintas (la presión ronda los 990 mbar y la temperatura los 10 °C). Restamos la media y dividimos entre la desviación estándar de cada columna, calculadas **solo con el entrenamiento** (para no usar información del futuro).

In [ ]:
n = len(df)
train_end, valid_end = int(n * 0.70), int(n * 0.85)
mean = df.iloc[:train_end].mean()
std = df.iloc[:train_end].std()
data = torch.tensor(((df - mean) / std).values, dtype=torch.float32)

target_col = df.columns.get_loc("T (degC)")
t_std = std["T (degC)"]  # para pasar los errores de vuelta a °C
n_features = data.shape[1]

train_data, valid_data, test_data = data[:train_end], data[train_end:valid_end], data[valid_end:]
print("Horas de entrenamiento, validación y prueba:", len(train_data), len(valid_data), len(test_data))
print("Variables por paso:", n_features)

**Ventanas.** Cada ejemplo toma las últimas **72 horas** (3 días) con las 14 variables, y la etiqueta es la temperatura **24 horas después** de la última hora de la ventana. La forma de cada entrada es `(72, 14)`: **(pasos, variables por paso)**.

In [ ]:
class WeatherWindowDataset(Dataset):
    def __init__(self, data, window_length=72, horizon=24):
        self.data = data
        self.window_length = window_length
        self.horizon = horizon

    def __len__(self):
        return len(self.data) - self.window_length - self.horizon + 1

    def __getitem__(self, idx):
        end = idx + self.window_length
        window = self.data[idx:end]                                    # (72, 14)
        target = self.data[end + self.horizon - 1, target_col:target_col + 1]  # temperatura 24 h después
        return window, target


train_loader = DataLoader(WeatherWindowDataset(train_data), batch_size=256, shuffle=True)
valid_loader = DataLoader(WeatherWindowDataset(valid_data), batch_size=256)
test_loader = DataLoader(WeatherWindowDataset(test_data), batch_size=256)

x, y = next(iter(train_loader))
print("Entrada:", tuple(x.shape), " Etiqueta:", tuple(y.shape))

**Pronóstico de referencia.** El pronóstico ingenuo para el clima es: *"mañana a esta hora hará la misma temperatura que ahora"*. Es el número que nuestra red debe superar. Lo medimos con el MAE en grados Celsius.

In [ ]:
def naive_mae(loader):
    total, n = 0.0, 0
    for X, y in loader:
        y_naive = X[:, -1, target_col:target_col + 1]   # temperatura de la última hora de la ventana
        total += (y_naive - y).abs().sum().item()
        n += len(y)
    return total / n * t_std

print(f"MAE ingenuo en validación: {naive_mae(valid_loader):.2f} °C")

## Etapa 3: La clase para entrenar

Es la misma organización de la clase `RidershipForecaster` de la Experiencia: arma la red en `__init__` (aquí, `TemperatureNet`, la red que ustedes definen en la Etapa 4) y tiene `train_epoch`, `evaluate`, `fit` y `predict`. Pérdida `HuberLoss`, optimizador Adam, MAE como métrica (aquí en °C) y se guarda el modelo con mejor MAE de validación.

In [ ]:
class WeatherForecaster:
    def __init__(self, n_features, lr=1e-3):
        # La red se arma dentro de la clase: TemperatureNet, la que ustedes definen en la Etapa 4
        self.net = TemperatureNet(n_features).to(device)
        self.loss_fn = nn.HuberLoss()
        self.optimizer = torch.optim.Adam(self.net.parameters(), lr=lr)
        self.scale = t_std  # para reportar el MAE en °C
        self.history = {"loss": [], "mae": [], "val_loss": [], "val_mae": []}

    def train_epoch(self, loader):
        # Una época: recorre todos los minibatches y actualiza los pesos
        self.net.train()
        total_loss, total_abs, n = 0.0, 0.0, 0
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            y_pred = self.net(X)
            loss = self.loss_fn(y_pred, y)
            self.optimizer.zero_grad()
            loss.backward()
            self.optimizer.step()
            total_loss += loss.item() * len(X)
            total_abs += (y_pred - y).abs().sum().item()
            n += y.numel()
        return total_loss / len(loader.dataset), total_abs / n * self.scale

    def evaluate(self, loader):
        # Mide el desempeño sin entrenar (sin calcular gradientes)
        self.net.eval()
        total_loss, total_abs, n = 0.0, 0.0, 0
        with torch.no_grad():
            for X, y in loader:
                X, y = X.to(device), y.to(device)
                y_pred = self.net(X)
                total_loss += self.loss_fn(y_pred, y).item() * len(X)
                total_abs += (y_pred - y).abs().sum().item()
                n += y.numel()
        return total_loss / len(loader.dataset), total_abs / n * self.scale

    def fit(self, train_loader, valid_loader, epochs=10):
        best_mae, best_state = float("inf"), None
        pbar = tqdm(range(epochs), desc="Entrenamiento", unit="época")
        for epoch in pbar:
            loss, mae = self.train_epoch(train_loader)
            val_loss, val_mae = self.evaluate(valid_loader)
            for key, value in zip(self.history, (loss, mae, val_loss, val_mae)):
                self.history[key].append(value)
            if val_mae < best_mae:  # guardamos los mejores pesos
                best_mae, best_state = val_mae, copy.deepcopy(self.net.state_dict())
            pbar.set_postfix(mae=f"{mae:.2f}", val_mae=f"{val_mae:.2f}", mejor_val_mae=f"{best_mae:.2f}")
        self.net.load_state_dict(best_state)  # nos quedamos con los mejores pesos
        print(f"Mejor MAE de validación: {best_mae:.2f}")
        return best_mae

    def predict(self, X):
        # Devuelve el pronóstico (en la escala de entrenamiento) para cada ventana
        self.net.eval()
        with torch.no_grad():
            return self.net(X.to(device)).cpu()

## Etapa 4: Definición del modelo

Definan aquí la red recurrente que propusieron en la Reflexión. Recuerden de la Experiencia:

- La entrada tiene forma `(batch, 72, 14)`: ahora `input_size` es el número de variables por paso, `n_features`.
- Para pronosticar un solo valor, se usa el estado oculto del **último paso** como resumen de la ventana, seguido de una capa `nn.Linear`.

In [ ]:
class TemperatureNet(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        # TODO: definan aquí las capas de su red, según la hipótesis de la Reflexión.
        # Pistas:
        #   - Una capa recurrente: nn.RNN, nn.LSTM o nn.GRU, con input_size=n_features
        #     y batch_first=True. Decidan hidden_size y num_layers.
        #   - Una capa nn.Linear al final que entregue 1 número (la temperatura).
        pass

    def forward(self, X):
        # X tiene forma (batch, pasos, n_features)
        # TODO: pasen X por la capa recurrente, tomen el estado oculto del último paso
        #       y conviértanlo en la predicción con la capa lineal. Debe devolver (batch, 1).
        pass

In [ ]:
# Verificamos que la red entregue una salida de forma (batch, 1) y contamos sus parámetros
model = TemperatureNet(n_features)
print(model)
print("Salida:", tuple(model(x).shape))
print("Parámetros:", sum(p.numel() for p in model.parameters()))

## Etapa 5: Entrenamiento y evaluación del modelo

Entrenamos la red. Pueden cambiar la tasa de aprendizaje (`lr`) y el número de épocas.

In [ ]:
torch.manual_seed(42)
forecaster = WeatherForecaster(n_features, lr=1e-3)
forecaster.fit(train_loader, valid_loader, epochs=10);

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(forecaster.history["mae"], label="entrenamiento")
ax.plot(forecaster.history["val_mae"], label="validación")
ax.axhline(naive_mae(valid_loader), color="gray", ls=":", label="ingenuo (validación)")
ax.set_xlabel("Época"); ax.set_ylabel("MAE (°C)"); ax.grid(); ax.legend()
plt.show()

Finalmente, medimos el modelo en el conjunto de prueba y lo comparamos con el pronóstico ingenuo:

In [ ]:
_, test_mae = forecaster.evaluate(test_loader)
print(f"MAE de prueba de la red:   {test_mae:.2f} °C")
print(f"MAE de prueba del ingenuo: {naive_mae(test_loader):.2f} °C")

In [ ]:
# Pronósticos de la red sobre dos semanas del conjunto de prueba
test_set = WeatherWindowDataset(test_data)
idx = range(0, 24 * 14)
X_plot = torch.stack([test_set[i][0] for i in idx])
y_real = torch.stack([test_set[i][1] for i in idx]).squeeze() * t_std + mean["T (degC)"]
y_pred = forecaster.predict(X_plot).squeeze() * t_std + mean["T (degC)"]

plt.figure(figsize=(11, 4))
plt.plot(y_real, label="real")
plt.plot(y_pred, label="pronóstico a 24 horas")
plt.xlabel("Horas"); plt.ylabel("°C"); plt.grid(); plt.legend()
plt.show()